# 26-Class ECG Classifier V3.2 (PTB-XL + Chapman + Challenge + CODE-15%)

Trains ECGNetJoint on **~457K records** with transfer learning from V3 best (AUROC=0.9682).

**Requires tars pre-built and synced to Drive (see CLAUDE.md).**

---
| Cell | Runtime | Time | What |
|------|---------|------|------|
| 1 | **GPU/TPU** | ~30-40 min | Restore all 4 datasets + setup |
| 2 | **GPU/TPU** | ~40-90 min (TPU) | Train V3.2 + save model to Drive |

**Why Drive API streaming?**  
Colab's Drive FUSE mount caches files to local disk as they're read.  
The EKG Drive folder is ~96 GB, which fills the 107 GB Colab disk.  
Streaming via Drive API bypasses that cache — only extracted files land on disk.


In [ ]:
# -----------------------------------------------------------------------------
# Cell 1 -- GPU/TPU runtime | Setup + restore all data from Drive tars
#
# Tars required in Drive/EKG/ekg_datasets/:
#   code15.tar  (~68 GB)   challenge2021.tar (~7 GB)
#   chapman.tar (~5.5 GB)  ptbxl.tar         (~2.7 GB)
#
# ALL tars are streamed via Drive API (not FUSE mount) to avoid filling
# local disk with the FUSE cache. Only extracted files (~80 GB) land on SSD.
#
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 1: GPU/TPU setup + restore data')
print('=' * 60)

# -- Verify accelerator ---------------------------------------------------
import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Runtime > Change runtime type > T4 GPU or TPU v6e')
print(f'Accelerator: {accel}')

# TPU: install torch_xla if needed (via subprocess only -- never import it here)
if accel == 'TPU':
    if subprocess.run([sys.executable, '-c', 'import torch_xla'],
                      capture_output=True).returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch_xla'], check=True)
        print('torch_xla installed')

# -- Mount Drive (for scripts + model only -- tars streamed via API) ------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn', 'h5py'], check=True)

import shutil as _sh
free_gb = _sh.disk_usage('/content').free / 1e9
print(f'Deps ready | SSD free: {free_gb:.0f} GB (need ~80 GB for extracted files) ({time.time()-t0:.0f}s)')

# -- Copy scripts from Drive ----------------------------------------------
SCRIPTS_DIR  = Path('/content/drive/MyDrive/EKG')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

for s in ['multilabel_v3.py', 'multilabel_classifier.py', 'cnn_classifier.py',
          'dataset_chapman.py', 'dataset_challenge.py', 'dataset_code15.py']:
    src = SCRIPTS_DIR / s
    if not src.exists():
        raise FileNotFoundError(f'{s} not found at {SCRIPTS_DIR}/')
    shutil.copy(src, f'/content/{s}')
print(f'Scripts copied ({time.time()-t0:.0f}s)')

# -- Drive API auth -------------------------------------------------------
# authenticate_user() after drive.mount() is silent (no dialog) -- drive.mount()
# already did OAuth. This call just writes the ADC file so google.auth.default()
# can find the existing credentials.
from google.colab import auth as _colab_auth
_colab_auth.authenticate_user()
import google.auth
from google.auth.transport.requests import Request as _GReq
from googleapiclient.discovery import build as _build
import requests as _rq

_creds, _ = google.auth.default()
_creds.refresh(_GReq())
_svc = _build('drive', 'v3', cache_discovery=False)

def _folder_id(parent, name):
    q = (f"name='{name}' and "
         f"mimeType='application/vnd.google-apps.folder' and "
         f"'{parent}' in parents and trashed=false")
    r = _svc.files().list(q=q, fields='files(id)').execute()
    f = r.get('files', [])
    return f[0]['id'] if f else None

def _file_info(parent, name):
    q = f"name='{name}' and '{parent}' in parents and trashed=false"
    r = _svc.files().list(q=q, fields='files(id,size)').execute()
    f = r.get('files', [])
    if not f: return None, 0
    return f[0]['id'], int(f[0].get('size', 0))

_ekg_id  = _folder_id('root', 'EKG')
_data_id = _folder_id(_ekg_id, 'ekg_datasets')
print(f'Drive API ready  (EKG folder found: {_ekg_id is not None})')

def stream_tar(fid, target, label, size_gb):
    """Stream tar from Drive API -> tar stdin. No FUSE caching, only extracted files on disk."""
    _creds.refresh(_GReq())  # refresh token before each large download
    print(f'{label}: streaming {size_gb:.1f} GB (Drive API)...', flush=True)
    url = f'https://www.googleapis.com/drive/v3/files/{fid}?alt=media'
    hdrs = {'Authorization': f'Bearer {_creds.token}'}
    proc = subprocess.Popen(
        ['tar', 'xf', '-', '-C', target],
        stdin=subprocess.PIPE, stderr=subprocess.PIPE
    )
    with _rq.get(url, headers=hdrs, stream=True, timeout=(30, 600)) as resp:
        resp.raise_for_status()
        for chunk in resp.iter_content(chunk_size=128 << 20):  # 128 MB chunks
            if chunk:
                proc.stdin.write(chunk)
    proc.stdin.close()
    _, err = proc.communicate()
    if proc.returncode:
        raise RuntimeError(
            f'{label}: tar failed (exit {proc.returncode})\n'
            f'stderr: {err.decode()[:400]}'
        )
    print(f'{label}: done ({time.time()-t0:.0f}s)', flush=True)

# -- Restore datasets (largest first) ------------------------------------

# [1/4] CODE-15% (~68 GB tar, ~64 GB extracted)
code15_dir = Path('/content/ekg_datasets/code15')
n_h5 = sum(1 for i in range(18)
           if (code15_dir / 'raw' / f'exams_part{i}.h5').exists())
if n_h5 == 18:
    print(f'CODE-15%: {n_h5}/18 H5 files on SSD -- skipping')
else:
    fid, sz = _file_info(_data_id, 'code15.tar')
    if not fid:
        raise FileNotFoundError(
            'code15.tar not found in Drive/EKG/ekg_datasets/.\n'
            'Build locally: run run_code15_after_download.bat, then let Drive finish syncing.'
        )
    stream_tar(fid, '/content/ekg_datasets', 'CODE-15%', sz / 1e9)
    n_h5 = sum(1 for i in range(18)
               if (code15_dir / 'raw' / f'exams_part{i}.h5').exists())
    print(f'CODE-15%: {n_h5}/18 H5 files on SSD')

# [2/4] Challenge 2021 (~7 GB) -- optional
challenge_dir = Path('/content/ekg_datasets/challenge2021')
if len(list(challenge_dir.rglob('*.mat'))) >= 50000:
    print('Challenge: already on SSD -- skipping')
else:
    fid, sz = _file_info(_data_id, 'challenge2021.tar')
    if fid:
        stream_tar(fid, '/content/ekg_datasets', 'Challenge', sz / 1e9)
    else:
        print('WARNING: challenge2021.tar not on Drive -- skipping (optional)')

# [3/4] Chapman (~5.5 GB, paths inside: ekg_datasets/... -> extract -C /content)
chapman_dir = Path('/content/ekg_datasets/chapman')
if len(list(chapman_dir.rglob('*.mat'))) >= 44000:
    print('Chapman: already on SSD -- skipping')
else:
    fid, sz = _file_info(_data_id, 'chapman.tar')
    if not fid:
        raise FileNotFoundError('chapman.tar not found in Drive/EKG/ekg_datasets/.')
    stream_tar(fid, '/content', 'Chapman', sz / 1e9)

# [4/4] PTB-XL (~2.7 GB)
ptbxl_dir = Path('/content/ekg_datasets/ptbxl')
if len(list(ptbxl_dir.rglob('*.dat'))) >= 18000:
    print('PTB-XL: already on SSD -- skipping')
else:
    fid, sz = _file_info(_data_id, 'ptbxl.tar')
    if not fid:
        raise FileNotFoundError('ptbxl.tar not found in Drive/EKG/ekg_datasets/.')
    stream_tar(fid, '/content/ekg_datasets', 'PTB-XL', sz / 1e9)

# -- Restore v2 model for transfer learning ------------------------------
v2_dst = Path('/content/models/ecg_multilabel_v2.pt')
if not v2_dst.exists():
    v2_src = DRIVE_MODELS / 'ecg_multilabel_v2.pt'
    if v2_src.exists():
        shutil.copy(v2_src, v2_dst)
        print('v2 model copied for transfer learning')
    else:
        print('WARNING: v2 model not found -- training from scratch')

free_after = _sh.disk_usage('/content').free / 1e9
print(f'\nCell 1 done ({time.time()-t0:.0f}s) | SSD free: {free_after:.0f} GB | ready to train')

In [ ]:
# -----------------------------------------------------------------------------
# Cell 2 -- GPU/TPU runtime | Train V3.2 + save to Drive
# TPU (v6e): ~40-90 min  |  GPU (T4): ~2-4 hrs
# First TPU epoch is slow (XLA compilation) -- this is normal.
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 2: Train V3.2 + save model')
print('=' * 60)
os.chdir('/content')

import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Run Cell 1 first.')

if accel == 'TPU':
    try:
        fd = os.open('/dev/vfio/0', os.O_RDONLY | os.O_NONBLOCK)
        os.close(fd)
    except OSError as e:
        raise RuntimeError(
            f'TPU device busy ({e}). Runtime > Restart runtime, then re-run Cell 1 + 2.'
        )

print(f'Accelerator: {accel}')
print('Starting multilabel_v3.py...', flush=True)
print('-' * 60, flush=True)

proc = subprocess.Popen(
    [sys.executable, '-u', 'multilabel_v3.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('-' * 60, flush=True)
if proc.returncode != 0:
    raise RuntimeError(f'Training failed (exit {proc.returncode})')

src = Path('/content/models/ecg_multilabel_v3.pt')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
if src.exists():
    dst = DRIVE_MODELS / 'ecg_multilabel_v3.pt'
    shutil.copy(src, dst)
    print(f'Model saved to Drive ({dst.stat().st_size/1e6:.0f} MB)')
    print('Syncs to local PC via Google Drive for Desktop.')
else:
    print('WARNING: model file not found')

print(f'\nCell 2 done ({time.time()-t0:.0f}s)')
